<a href="https://colab.research.google.com/github/melisa176/phishing-detection/blob/main/notebooks/05_entrenamiento_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Entrenamiento en inglés (mBERT, XLM-RoBERTa)

Este notebook entrena dos modelos multilingües sobre el conjunto de
entrenamiento en inglés (Enron + Nazario + phishing_pot, ya limpio y
balanceado): mBERT y XLM-RoBERTa. BETO no se incluye aquí, ya que es un
modelo monolingüe en español, reservado para el notebook 06.

La evaluación final fuera de distribución contra SpaPhish y
SpearPhishMX se realiza por separado, en el notebook 07.

In [ ]:
!pip install transformers datasets scikit-learn accelerate -q

# ENTRENAR mBERT EN INGLÉS

## 1. Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print('Dispositivo:', device)

## 2. Conexión a Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Carga del conjunto de entrenamiento en inglés

In [ ]:
RUTA_DATASET = '/content/drive/MyDrive/dataset_limpio_ingles_balanceado.csv'

df = pd.read_csv(RUTA_DATASET)
print('Filas cargadas:', len(df))
print('Columnas:', df.columns.tolist())
print()
print('Distribución de etiquetas:')
print(df['label'].value_counts())

## 4. División en entrenamiento, validación y prueba

División estratificada (80/10/10), de forma que la proporción de
clases se mantenga igual en las tres particiones.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp['label'], random_state=42
)

print('Entrenamiento:', len(df_train))
print('Validación:', len(df_val))
print('Prueba:', len(df_test))

## 5. Cálculo de pesos de clase

El conjunto ya está balanceado desde la etapa de limpieza, pero se
calculan los pesos de clase como práctica robusta, para que el
entrenamiento siga siendo correcto incluso si en el futuro se usa este
mismo notebook con un dataset que no esté perfectamente balanceado.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

pesos_clase = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_train['label']),
    y=df_train['label']
)
pesos_clase_tensor = torch.tensor(pesos_clase, dtype=torch.float).to(device)

print('Pesos de clase calculados:', pesos_clase)

## 6. Función de entrenamiento

Se define una función reutilizable que recibe el nombre del modelo
preentrenado y devuelve el modelo ajustado junto con sus métricas sobre
el conjunto de prueba interno. Usa los pesos de clase calculados en el
paso anterior dentro de una función de pérdida personalizada, en vez de
la pérdida por defecto del `Trainer`.

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch.nn as nn


def calcular_metricas(eval_pred):
    logits, labels = eval_pred
    predicciones = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predicciones, average='binary'
    )
    accuracy = accuracy_score(labels, predicciones)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


class EntrenadorConPesos(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        funcion_perdida = nn.CrossEntropyLoss(weight=pesos_clase_tensor)
        loss = funcion_perdida(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## 7. Entrenamiento de mBERT

In [ ]:
nombre_modelo = 'bert-base-multilingual-cased'
nombre_corto = 'mbert_es'

tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

def tokenizar(ejemplos):
    return tokenizer(ejemplos['text'], truncation=True, padding='max_length', max_length=512)

ds_train = Dataset.from_pandas(df_train[['text', 'label']]).map(tokenizar, batched=True)
ds_val = Dataset.from_pandas(df_val[['text', 'label']]).map(tokenizar, batched=True)
ds_test = Dataset.from_pandas(df_test[['text', 'label']]).map(tokenizar, batched=True)

modelo_mbert = AutoModelForSequenceClassification.from_pretrained(nombre_modelo, num_labels=2)

In [ ]:
args_entrenamiento = TrainingArguments(
    output_dir=f'./resultados_{nombre_corto}',
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir=f'./logs_{nombre_corto}',
    report_to='none',
)

entrenador_mbert = EntrenadorConPesos(
    model=modelo_mbert,
    args=args_entrenamiento,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=calcular_metricas,
)

entrenador_mbert.train()

## 8. Evaluación de mBERT sobre el conjunto de prueba interno

In [ ]:
metricas_mbert = entrenador_mbert.evaluate(ds_test)
print('--- Métricas de mBERT en prueba interna ---')
print(metricas_mbert)

## 9. Guardado del modelo mBERT entrenado

In [ ]:
entrenador_mbert.save_model('/content/drive/MyDrive/modelo_mbert_en')
print('Modelo mBERT guardado en Drive.')

# ENTRENAR XLM-ROBERTA EN INGLÉS